TRANSFORMACION DE DATOS Y REDUCCION DE FILAS A ENCUENTROS COMPETITIVOS

In [22]:
import pandas as pd
df = pd.read_csv('results.csv')
df.head()


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False


In [23]:
#Convertir la columna date en formato datetime
df["date"]=pd.to_datetime(df["date"])

In [24]:
#Solo queremos conservar los partidos unicamente desde 1990 en adelante
df_actual=df[df["date"].dt.year >= 1990]

In [25]:
#Eliminamos los partidos amistosos, solo se usaran los partidos oficiales
df_oficial=df_actual[df_actual["tournament"]!="Friendly"].copy()


In [26]:
#Creamos un identificador unico para cada partido
df_oficial["match_id"]=range(len(df_oficial))


In [27]:
print(f"Dataset filtrado con éxito. Total de partidos competitivos: {len(df_oficial)}")

Dataset filtrado con éxito. Total de partidos competitivos: 21530


DEFINIR EL TARGET Y TRADUCIR LOS EQUIPOS A NUMEROS

In [28]:
import numpy as np
from sklearn.preprocessing import LabelEncoder  

In [29]:
#Creamos las variables objetivo
#Gana Local = 1, Empate = 0, Gana Visitante = 2
condiciones=[
    (df_oficial["home_score"] > df_oficial["away_score"]),
    (df_oficial["home_score"] == df_oficial["away_score"]),
    (df_oficial["home_score"] < df_oficial["away_score"])   
]
complementos=[1,2,0]
df_oficial["target"]=np.select(condiciones,complementos)

In [30]:
# Inicializamos el codificador para traducir texto de países a números de ID
encoder=LabelEncoder()
equipos=pd.concat([df_oficial["home_team"],df_oficial["away_team"]]).unique()
encoder.fit(equipos)

LabelEncoder()

In [31]:
#Aplicamos la transformación creando columnas numéricas de ID
df_oficial['home_team_code'] = encoder.transform(df_oficial['home_team'])
df_oficial['away_team_code'] = encoder.transform(df_oficial['away_team'])

In [32]:
print("Variable objetivo construida y equipos codificados numéricamente.")

Variable objetivo construida y equipos codificados numéricamente.


In [33]:
# 1. Separamos los datos desde la perspectiva del Local
locales = df_oficial[['match_id', 'date', 'home_team', 'home_score', 'away_score']].copy()
locales.columns = ['match_id', 'date', 'team', 'goles_favor', 'goles_contra']
locales['puntos'] = np.select([locales['goles_favor'] > locales['goles_contra'], locales['goles_favor'] == locales['goles_contra']], [3, 1], default=0)

# 2. Separamos los datos desde la perspectiva del Visitante
visitantes = df_oficial[['match_id', 'date', 'away_team', 'away_score', 'home_score']].copy()
visitantes.columns = ['match_id', 'date', 'team', 'goles_favor', 'goles_contra']
visitantes['puntos'] = np.select([visitantes['goles_favor'] > visitantes['goles_contra'], visitantes['goles_favor'] == visitantes['goles_contra']], [3, 1], default=0)

# 3. Unimos ambas perspectivas en un historial unificado y ordenado cronológicamente por país
historial = pd.concat([locales, visitantes]).sort_values(['team', 'date'])

# 4. Calculamos las métricas móviles de los últimos 5 partidos (restando el partido actual con shift)
historial['racha_5_partidos'] = historial.groupby('team')['puntos'].transform(lambda x: x.shift(1).rolling(5, min_periods=1).sum()).fillna(0)
historial['promedio_goles_favor'] = historial.groupby('team')['goles_favor'].transform(lambda x: x.shift(1).rolling(5, min_periods=1).mean()).fillna(0)
historial['promedio_goles_contra'] = historial.groupby('team')['goles_contra'].transform(lambda x: x.shift(1).rolling(5, min_periods=1).mean()).fillna(0)

# 5. Reincorporamos las métricas al dataset principal mediante cruces de tablas (Merge)
df_oficial = df_oficial.merge(historial[['match_id', 'team', 'racha_5_partidos', 'promedio_goles_favor', 'promedio_goles_contra']], left_on=['match_id', 'home_team'], right_on=['match_id', 'team'], how='left').rename(columns={'racha_5_partidos': 'racha_local', 'promedio_goles_favor': 'goles_favor_local', 'promedio_goles_contra': 'goles_contra_local'}).drop(columns=['team'])
df_oficial = df_oficial.merge(historial[['match_id', 'team', 'racha_5_partidos', 'promedio_goles_favor', 'promedio_goles_contra']], left_on=['match_id', 'away_team'], right_on=['match_id', 'team'], how='left').rename(columns={'racha_5_partidos': 'racha_visitante', 'promedio_goles_favor': 'goles_favor_visitante', 'promedio_goles_contra': 'goles_contra_visitante'}).drop(columns=['team'])

df_ready = df_oficial.dropna().copy()
print("Variables de rendimiento (goles y puntos recientes) consolidadas en el dataset principal.")

Variables de rendimiento (goles y puntos recientes) consolidadas en el dataset principal.


ENTRENAMIENTO DEL MODELO

In [34]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# 1. Definimos las columnas que usará el modelo para predecir
features = [
    'home_team_code', 'away_team_code', 
    'racha_local', 'racha_visitante',
    'goles_favor_local', 'goles_contra_local',
    'goles_favor_visitante', 'goles_contra_visitante'
]

X = df_ready[features]
y = df_ready['target']

# 2. Segmentamos los conjuntos de entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

# 3. Inicializamos y entrenamos el Bosque Aleatorio
modelo_mundial = RandomForestClassifier(n_estimators=100, random_state=42)
modelo_mundial.fit(X_train, y_train)

# 4. Validamos la precisión del modelo entrenado localmente
predicciones = modelo_mundial.predict(X_test)
print(f"¡Modelo entrenado con éxito en tu computadora!")
print(f"Precisión lograda en validación: {accuracy_score(y_test, predicciones) * 100:.2f}%")

¡Modelo entrenado con éxito en tu computadora!
Precisión lograda en validación: 54.09%


PREDECIR PARTIDOS

In [35]:
# 1. Filtramos los partidos de HOY buscando en el calendario completo (df_oficial)
partidos_hoy = df_oficial[df_oficial['date'] == '2026-06-14'].copy()

# 2. Generamos las predicciones con tu modelo ya entrenado
partidos_hoy['pred_codigo'] = modelo_mundial.predict(partidos_hoy[features])

# 3. Traducimos los resultados numéricos a texto
condiciones_hoy = [
    partidos_hoy['pred_codigo'] == 1,
    partidos_hoy['pred_codigo'] == 2,
    partidos_hoy['pred_codigo'] == 0
]
textos_hoy = [
    'Gana ' + partidos_hoy['home_team'],
    'Gana ' + partidos_hoy['away_team'],
    'Empate'
]
partidos_hoy['prediccion_final'] = np.select(condiciones_hoy, textos_hoy)

# 4. Mostramos la cartelera de los partidos de hoy
partidos_hoy[['date', 'home_team', 'away_team', 'prediccion_final']]

,date,home_team,away_team,prediccion_final
21466,2026-06-14,Germany,Curaçao,Gana Germany
21467,2026-06-14,Ivory Coast,Ecuador,Gana Ivory Coast
21468,2026-06-14,Netherlands,Japan,Empate
21469,2026-06-14,Sweden,Tunisia,Gana Sweden


ACTUALIZACIÓN DE LOS RESULTADOS

In [ ]:
# --- RESULTADOS DEL VIERNES 12 DE JUNIO ---

# Canadá 1 - 1 Bosnia
df.loc[(df['date'] == '2026-06-12') & (df['home_team'] == 'Canada'), 'home_score'] = 1
df.loc[(df['date'] == '2026-06-12') & (df['away_team'] == 'Bosnia and Herzegovina'), 'away_score'] = 1

# Estados Unidos 4 - 1 Paraguay
df.loc[(df['date'] == '2026-06-12') & (df['home_team'] == 'United States'), 'home_score'] = 4
df.loc[(df['date'] == '2026-06-12') & (df['away_team'] == 'Paraguay'), 'away_score'] = 1

# --- RESULTADOS DEL SÁBADO 13 DE JUNIO (AYER) ---

# Catar 1 - 1 Suiza
df.loc[(df['date'] == '2026-06-13') & (df['home_team'] == 'Qatar'), 'home_score'] = 1
df.loc[(df['date'] == '2026-06-13') & (df['away_team'] == 'Switzerland'), 'away_score'] = 1

# Brasil 1 - 1 Marruecos
df.loc[(df['date'] == '2026-06-13') & (df['home_team'] == 'Brazil'), 'home_score'] = 1
df.loc[(df['date'] == '2026-06-13') & (df['away_team'] == 'Morocco'), 'away_score'] = 1

# Haití 0 - 1 Escocia
df.loc[(df['date'] == '2026-06-13') & (df['home_team'] == 'Haiti'), 'home_score'] = 0
df.loc[(df['date'] == '2026-06-13') & (df['away_team'] == 'Scotland'), 'away_score'] = 1

# Australia 2 - 0 Turquía
df.loc[(df['date'] == '2026-06-13') & (df['home_team'] == 'Australia'), 'home_score'] = 2
df.loc[(df['date'] == '2026-06-13') & (df['away_team'] == 'Turkey'), 'away_score'] = 0

# Alemania 7 - 1 Curazao
df.loc[(df['date'] == '2026-06-14') & (df['home_team'] == 'Germany'), 'home_score'] = 7
df.loc[(df['date'] == '2026-06-14') & (df['away_team'] == 'Curaçao'), 'away_score'] = 1

# Paises Bajos 2 - 2 Japon
df.loc[(df['date'] == '2026-06-14') & (df['home_team'] == 'Netherlands'), 'home_score'] = 2
df.loc[(df['date'] == '2026-06-14') & (df['away_team'] == 'Japan'), 'away_score'] = 2

# Costa de Marfil 1 - 0 Ecuador
df.loc[(df['date'] == '2026-06-14') & (df['home_team'] == 'Ivory Coast'), 'home_score'] = 1
df.loc[(df['date'] == '2026-06-14') & (df['away_team'] == 'Ecuador'), 'away_score'] = 0


print("Resultados cargados en la matriz")

Resultados cargados en la matriz


INSPECCION VISUAL

In [37]:
# Filtramos la tabla para ver únicamente los partidos del 12 y 13 de junio
fechas_buscadas = ['2026-06-12', '2026-06-13']
comprobacion = df[df['date'].isin(fechas_buscadas)]

# Mostramos las columnas clave
comprobacion[['date', 'home_team', 'away_team', 'home_score', 'away_score']]

C:\Users\ELVA\AppData\Local\Temp\ipykernel_11080\3829532250.py:3: FutureWarning: The behavior of 'isin' with dtype=datetime64[ns] and castable values (e.g. strings) is deprecated. In a future version, these will not be considered matching by isin. Explicitly cast to the appropriate dtype before calling isin instead.
  comprobacion = df[df['date'].isin(fechas_buscadas)]


,date,home_team,away_team,home_score,away_score
49407,2026-06-12,Canada,Bosnia and Herzegovina,1.0,1.0
49408,2026-06-12,United States,Paraguay,4.0,1.0
49409,2026-06-13,Qatar,Switzerland,1.0,1.0
49410,2026-06-13,Brazil,Morocco,1.0,1.0
49411,2026-06-13,Haiti,Scotland,0.0,1.0
49412,2026-06-13,Australia,Turkey,2.0,0.0


PRESICIÓN ACTUALIZADA

In [38]:
predicciones_prueba = modelo_mundial.predict(X_test)
precision_actual = accuracy_score(y_test, predicciones_prueba) * 100

print(f"La precisión actual es de: {precision_actual:.2f}%")

La precisión actual es de: 54.09%


INTERPRETACION DE RESULTADOS DE MANERA VISUAL

In [39]:
# 1. Filtramos TODOS los partidos desde hoy en adelante usando la tabla maestra
partidos_futuros = df_oficial[df_oficial['date'] >= '2026-06-14'].copy()

# 2. Generamos las predicciones para todo el resto del torneo
partidos_futuros['pred_codigo'] = modelo_mundial.predict(partidos_futuros[features])

# 3. Traducimos los resultados a texto
condiciones_vista = [
    partidos_futuros['pred_codigo'] == 1,
    partidos_futuros['pred_codigo'] == 2,
    partidos_futuros['pred_codigo'] == 0
]
textos_vista = [
    'Gana ' + partidos_futuros['home_team'],
    'Gana ' + partidos_futuros['away_team'],
    'Empate'
]
partidos_futuros['prediccion_final'] = np.select(condiciones_vista, textos_vista)

# 4. ¡LA EXPORTACIÓN! Guardamos la tabla completa en un CSV
partidos_futuros.to_csv('predicciones_mundial.csv', index=False)
print("Archivo CSV generado con éxito! Todo listo para el Dashboard.")

Archivo CSV generado con éxito! Todo listo para el Dashboard.
